# Need to Do

In [40]:
import geopandas as gpd
from gerrychain import Graph, Partition, updaters
from gerrychain.updaters import Tally, cut_edges
from gerrychain.proposals import recom
from gerrychain.constraints import within_percent_of_ideal_population
from gerrychain.optimization import SingleMetricOptimizer
from functools import partial
import numpy as np

In [48]:
pa_precincts = gpd.read_file("../data/cleaned/pa_cleaned_precincts") 
pa_graph = Graph.from_geodataframe(pa_precincts)
print(pa_precincts.columns.tolist())


['COUNTYFP', 'TOTPOP', 'VAP', 'BVAP', 'APBVAP', 'WVAP', 'HVAP', 'PRES20D', 'PRES20R', 'PRES16D', 'PRES16R', 'SENDIST', 'geometry']


In [49]:
my_updaters = {"population": updaters.Tally("TOTPOP", alias = "population"),
                   "VAP": updaters.Tally("VAP"),
                   "Any Percent Black VAP": updaters.Tally("APBVAP", alias = "Any Percent Black VAP")
              }

initial_partition = Partition(pa_graph, assignment = "SENDIST", updaters = my_updaters)

In [50]:
# gingles = Gingleator(initial_partition, pop_col = "TOTPOP",
#                      minority_perc_col = "BVAP")

# BURST_LEN = 5
# NUM_BURSTS = 100
# # testing to make sure it works with small size
# most_VRA, observed_scores = gingles.short_burst_run(
#     num_bursts = NUM_BURSTS, num_steps = BURST_LEN, verbose = True)
# np.save(f"results/PA_50dists_BVAP_sbl{BURST_LEN}.npy", observed_scores)

# # AI credit: I asked Claude to help me read gingleator.py
# # to help me understand what the functions and args do

In [51]:
POP_TOLERANCE = 0.05  # 5% — enacted plan deviates up to ~4.3% from ideal
NUM_STEPS = 50000
ideal_pop = sum(initial_partition["population"].values()) / len(initial_partition)

proposal = partial(
    recom,
    pop_col="TOTPOP",
    pop_target=ideal_pop,
    epsilon=POP_TOLERANCE,
    node_repeats=2,
)
chain_constraints = [within_percent_of_ideal_population(initial_partition, POP_TOLERANCE)]

In [73]:
def num_majority_black_districts(partition):
    districts = partition["population"].keys()
    majority_black_districts = 0
    for district in districts:
        district_population = partition["population"].get(district)
        black_population = partition["Any Percent Black VAP"].get(district)
        if(2 * black_population > district_population):
            majority_black_districts += 1
    return majority_black_districts
    # # I copied the layout of the short burstscode from the gerrychain docs
    # # Unfortunately they're optimizing for fewest number of cut edges, so
    # # it's trying to find the lowest score. So I'm inverting this number
    # # to try to fool the system, since I don't know how to tell it that a
    # # higher number is better
    # return -1 * majority_black_districts

optimizer = SingleMetricOptimizer(
    proposal=proposal,
    constraints=chain_constraints,
    initial_state=initial_partition,
    optimization_metric=num_majority_black_districts,
    maximize=True
)

In [74]:
burst_size = 10
total_bursts = 1000
total_steps = burst_size * total_bursts
# Short Bursts
min_scores = np.zeros(total_steps)
scores = np.zeros(total_steps)
for i, part in enumerate(optimizer.short_bursts(burst_size, total_bursts, with_progress_bar=True)):
    min_scores[i] = optimizer.best_score
    scores[i] = optimizer.score(part)

100%|██████████| 10000/10000 [02:42<00:00, 61.41it/s]


In [76]:
print(min_scores)
print(scores)

[0. 0. 0. ... 2. 2. 2.]
[0. 0. 0. ... 2. 2. 2.]
